In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets huggingface_hub

In [2]:
import os, json, random
import pandas as pd
import numpy as np
import requests
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_PATH = '/kaggle/working/datasets'  # ← fixed
os.makedirs(DATA_PATH, exist_ok=True)

# ── PILE ──────────────────────────────────────────────────────
print("[1/3] Loading PILE dataset (streaming)...")
pile_save_path = f'{DATA_PATH}/pile_50k.csv'

if os.path.exists(pile_save_path):
    print("Already exists")
    pile_df = pd.read_csv(pile_save_path)
else:
    parts = [
        {'skip': 0,      'target': 17000, 'label': 'Start'},
        {'skip': 200000, 'target': 17000, 'label': 'Middle'},
        {'skip': 500000, 'target': 16000, 'label': 'End'},
    ]
    all_samples = []
    for part in parts:
        print(f"  Collecting from {part['label']}...")
        dataset = load_dataset("monology/pile-uncopyrighted", split="train", streaming=True)
        samples, skipped, processed = [], 0, 0
        for example in dataset:
            if skipped < part['skip']:
                skipped += 1
                continue
            text = example.get('text', '').strip()
            if len(text) > 100:
                samples.append({'text': text, 'source': example.get('meta', {}).get('pile_set_name', 'unknown')})
                processed += 1
            if processed >= part['target']:
                break
        all_samples.extend(samples)
        print(f"  Collected {len(samples)}")
    random.shuffle(all_samples)
    pile_df = pd.DataFrame(all_samples[:50000])
    pile_df.to_csv(pile_save_path, index=False)
print(f"PILE ready: {len(pile_df)}")

# ── T-REx ─────────────────────────────────────────────────────
print("\n[2/3] Loading T-REx dataset...")
trex_save_path = f'{DATA_PATH}/trex_50k.csv'

if os.path.exists(trex_save_path):
    print("Already exists")
    trex_df = pd.read_csv(trex_save_path)
else:
    url = "http://dl.fbaipublicfiles.com/KILT/trex-train-kilt.jsonl"
    response = requests.get(url, stream=True)
    samples, raw_lines = [], 0
    for line in response.iter_lines():
        if len(samples) >= 50000: break
        if not line: continue
        raw_lines += 1
        try:
            item = json.loads(line)
            input_text = item.get('input', '').strip()
            outputs = item.get('output', [])
            if not outputs: continue
            answer = outputs[0].get('answer', '').strip()
            if not answer or '[SEP]' not in input_text: continue
            parts = input_text.split('[SEP]')
            subject  = parts[0].strip()
            relation = parts[1].strip() if len(parts) > 1 else ''
            samples.append({'text': f"{subject}'s {relation} is", 'entity': answer, 'subject': subject, 'relation': relation})
        except: continue
        if raw_lines % 100000 == 0:
            print(f"  Processed {raw_lines} lines, collected {len(samples)}...")
    response.close()
    trex_df = pd.DataFrame(samples[:50000])
    trex_df.to_csv(trex_save_path, index=False)
print(f"T-REx ready: {len(trex_df)}")

# ── MMLU ──────────────────────────────────────────────────────
print("\n[3/3] Loading MMLU dataset...")
mmlu_test_path = f'{DATA_PATH}/mmlu_test.csv'
mmlu_val_path  = f'{DATA_PATH}/mmlu_val.csv'

if os.path.exists(mmlu_test_path):
    print("Already exists")
    mmlu_df     = pd.read_csv(mmlu_test_path)
    mmlu_val_df = pd.read_csv(mmlu_val_path)
else:
    mmlu     = load_dataset("cais/mmlu", "all", split="test")
    mmlu_val = load_dataset("cais/mmlu", "all", split="validation")
    mmlu_df     = pd.DataFrame(mmlu)
    mmlu_val_df = pd.DataFrame(mmlu_val)
    mmlu_df.to_csv(mmlu_test_path, index=False)
    mmlu_val_df.to_csv(mmlu_val_path, index=False)
print(f"MMLU ready: {len(mmlu_df)}")

print("\nAll datasets ready!")

[1/3] Loading PILE dataset (streaming)...
Already exists
PILE ready: 50000

[2/3] Loading T-REx dataset...
Already exists
T-REx ready: 50000

[3/3] Loading MMLU dataset...
Already exists
MMLU ready: 14042

All datasets ready!


In [3]:
import os, json, random, ast, time
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ── Kaggle paths ───────────────────────────────────────────────
BASE_MODEL_PATH = "/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1"
PEFT_PATH       = "/kaggle/input/datasets/shahabahmad123/epoch3"
DATA_PATH       = "/kaggle/working/datasets"
os.makedirs(DATA_PATH, exist_ok=True)

# ── Model loading ──────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=quant_config,
    device_map="balanced"
)

print("Attaching PEFT adapter...")
model = PeftModel.from_pretrained(base_model, PEFT_PATH, is_trainable=False)
model.eval()
print("✅ Model loaded.")

# ── Load CSVs (must already exist in /kaggle/working/datasets) ─
pile_df     = pd.read_csv(f"{DATA_PATH}/pile_50k.csv")
trex_df     = pd.read_csv(f"{DATA_PATH}/trex_50k.csv")
mmlu_df     = pd.read_csv(f"{DATA_PATH}/mmlu_test.csv")

EVAL_CLM   = 5000
EVAL_FACTS = 5000
EVAL_MMLU  = 3000
ANSWER_OPTIONS = ["A", "B", "C", "D"]

def compute_ece(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(
            accuracies[mask].mean() - confidences[mask].mean()
        )
    return float(ece)

def compute_reliability_diagram_data(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        bin_sizes.append(int(mask.sum()))
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append(float((bins[i] + bins[i+1]) / 2))
        else:
            bin_accs.append(float(accuracies[mask].mean()))
            bin_confs.append(float(confidences[mask].mean()))
    return bin_accs, bin_confs, bin_sizes

pile_eval = pile_df.sample(n=EVAL_CLM,   random_state=SEED).reset_index(drop=True)
trex_eval = trex_df.sample(n=EVAL_FACTS, random_state=SEED).reset_index(drop=True)
mmlu_eval = mmlu_df.sample(n=EVAL_MMLU,  random_state=SEED).reset_index(drop=True)

print(f"✅ PILE eval  : {len(pile_eval)}")
print(f"✅ T-REx eval : {len(trex_eval)}")
print(f"✅ MMLU eval  : {len(mmlu_eval)}")

Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Attaching PEFT adapter...
✅ Model loaded.


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


✅ PILE eval  : 5000
✅ T-REx eval : 5000
✅ MMLU eval  : 3000


In [4]:
"""import time
from tqdm import tqdm

print("="*60)
print("TASK 1: CLM (PILE)")
print("="*60)

clm_confidences, clm_accuracies = [], []
clm_start = time.time()
processed = 0

pbar = tqdm(total=EVAL_CLM, desc="CLM Eval", unit="sample")

for idx, row in pile_eval.iterrows():
    text = str(row['text']).strip()
    try:
        input_ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)["input_ids"]
        seq_len = input_ids.shape[1]
        if seq_len < 3:
            continue
        pos        = np.random.randint(1, seq_len)
        context    = input_ids[:, :pos].to(model.device)
        true_token = input_ids[:, pos].item()
        with torch.no_grad():
            probs = F.softmax(model(input_ids=context).logits[:, -1, :].float(), dim=-1)
        clm_confidences.append(probs.max(dim=-1).values.item())
        clm_accuracies.append(int(probs.argmax(dim=-1).item() == true_token))
        processed += 1
        pbar.update(1)

        if processed % 500 == 0:
            elapsed  = (time.time() - clm_start) / 60
            eta      = (elapsed / processed) * (EVAL_CLM - processed)
            pbar.set_postfix({
                "Acc"    : f"{np.mean(clm_accuracies):.4f}",
                "ECE"    : f"{compute_ece(clm_confidences, clm_accuracies):.4f}",
                "Elapsed": f"{elapsed:.1f}m",
                "ETA"    : f"{eta:.1f}m"
            })
    except:
        continue

pbar.close()

clm_ece  = compute_ece(clm_confidences, clm_accuracies)
clm_acc  = float(np.mean(clm_accuracies))
clm_bins = compute_reliability_diagram_data(clm_confidences, clm_accuracies)
print(f"\n✅ CLM Done | ECE: {clm_ece:.4f} | Accuracy: {clm_acc:.4f}")"""

'import time\nfrom tqdm import tqdm\n\nprint("="*60)\nprint("TASK 1: CLM (PILE)")\nprint("="*60)\n\nclm_confidences, clm_accuracies = [], []\nclm_start = time.time()\nprocessed = 0\n\npbar = tqdm(total=EVAL_CLM, desc="CLM Eval", unit="sample")\n\nfor idx, row in pile_eval.iterrows():\n    text = str(row[\'text\']).strip()\n    try:\n        input_ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)["input_ids"]\n        seq_len = input_ids.shape[1]\n        if seq_len < 3:\n            continue\n        pos        = np.random.randint(1, seq_len)\n        context    = input_ids[:, :pos].to(model.device)\n        true_token = input_ids[:, pos].item()\n        with torch.no_grad():\n            probs = F.softmax(model(input_ids=context).logits[:, -1, :].float(), dim=-1)\n        clm_confidences.append(probs.max(dim=-1).values.item())\n        clm_accuracies.append(int(probs.argmax(dim=-1).item() == true_token))\n        processed += 1\n        pbar.update(1)\n\n    

In [7]:

from tqdm import tqdm
print("="*60)
print("TASK 3: MMLU (Multiple Choice)")
print("="*60)

mmlu_confidences, mmlu_accuracies = [], []
mmlu_start = time.time()
processed = 0

pbar = tqdm(total=EVAL_MMLU, desc="MMLU Eval", unit="sample")

for idx, row in mmlu_eval.iterrows():
    try:
        question = str(row['question']).strip()
        choices  = [str(row['choices'][i]) for i in range(4)] if isinstance(row['choices'], list) else ast.literal_eval(str(row['choices']))
        label    = int(row['answer'])  # 0,1,2,3 → A,B,C,D

        # build prompt
        prompt = (
            f"Question: {question}\n"
            f"A) {choices[0]}\n"
            f"B) {choices[1]}\n"
            f"C) {choices[2]}\n"
            f"D) {choices[3]}\n"
            f"Answer:"
        )

        input_ids = tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=512
        )["input_ids"].to(model.device)

        # get token ids for A B C D
        option_ids = [
            tokenizer(f" {opt}", add_special_tokens=False)["input_ids"][0]
            for opt in ANSWER_OPTIONS
        ]

        with torch.no_grad():
            probs = F.softmax(
                model(input_ids=input_ids).logits[:, -1, :].float(), dim=-1
            )

        # only look at probabilities of A B C D tokens
        option_probs = torch.tensor([probs[0, oid].item() for oid in option_ids])
        option_probs = option_probs / option_probs.sum()  # renormalize

        pred       = option_probs.argmax().item()
        confidence = option_probs.max().item()

        mmlu_confidences.append(confidence)
        mmlu_accuracies.append(int(pred == label))
        processed += 1
        pbar.update(1)

        if processed % 500 == 0:
            elapsed = (time.time() - mmlu_start) / 60
            eta     = (elapsed / processed) * (EVAL_MMLU - processed)
            pbar.set_postfix({
                "Acc"    : f"{np.mean(mmlu_accuracies):.4f}",
                "ECE"    : f"{compute_ece(mmlu_confidences, mmlu_accuracies):.4f}",
                "Elapsed": f"{elapsed:.1f}m",
                "ETA"    : f"{eta:.1f}m"
            })
    except:
        continue

pbar.close()

mmlu_ece  = compute_ece(mmlu_confidences, mmlu_accuracies)
mmlu_acc  = float(np.mean(mmlu_accuracies))
mmlu_bins = compute_reliability_diagram_data(mmlu_confidences, mmlu_accuracies)
print(f"\n✅ MMLU Done | ECE: {mmlu_ece:.4f} | Accuracy: {mmlu_acc:.4f}")

TASK 3: MMLU (Multiple Choice)


MMLU Eval: 100%|██████████| 3000/3000 [58:35<00:00,  1.17s/sample, Acc=0.2263, ECE=0.0237, Elapsed=58.6m, ETA=0.0m]   


✅ MMLU Done | ECE: 0.0237 | Accuracy: 0.2263


============================================================
TASK 1: CLM (PILE)
============================================================
  [1000/5000] (20.0%) | Time: 25.9m | ECE: 0.0326 | Acc: 0.6150
  [2000/5000] (40.0%) | Time: 51.3m | ECE: 0.0231 | Acc: 0.6105
  [3000/5000] (60.0%) | Time: 77.2m | ECE: 0.0183 | Acc: 0.6080
  [4000/5000] (80.0%) | Time: 104.0m | ECE: 0.0168 | Acc: 0.6102
  [5000/5000] (100.0%) | Time: 130.2m | ECE: 0.0201 | Acc: 0.6084

✅ CLM Done | ECE: 0.0201 | Accuracy: 0.6084




============================================================
TASK 2: FACT RETRIEVAL (T-REx)
============================================================
Facts Eval: 100%|██████████| 5000/5000 [53:19<00:00,  1.56sample/s, Acc=0.1224, ECE=0.0849, Elapsed=53.3m, ETA=0.0m] 
✅ Facts Done | ECE: 0.0849 | Accuracy: 0.1224